<a href="https://colab.research.google.com/github/Pulakspbl/Machine-learning-in-Python-with-scikit-learn/blob/main/Model_6.1_Ensemble/Exercise_M6_04_california_housing_hist_gradient_boosting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 Exercise M6.04

The aim of the exercise is to get familiar with the histogram
gradient-boosting in scikit-learn. Besides, we will use this model within a
cross-validation framework in order to inspect internal parameters found via
grid-search.

We will use the California housing dataset.

In [1]:
from sklearn.datasets import fetch_california_housing

data, target = fetch_california_housing(return_X_y=True, as_frame=True)
target *= 100  # rescale the target in k$

First, create a histogram gradient boosting regressor. You can set the trees number to be large, and configure the model to use early-stopping.


In [2]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split

# 2. Split the data into training and test sets
data, target = fetch_california_housing(return_X_y=True, as_frame=True)
target *= 100  # rescale the target in k$


# 3. Create the model with a large number of trees and early stopping
hist_model = HistGradientBoostingRegressor(
    max_iter=1000,         # Large number of trees
    early_stopping=True,   # Enable early stopping
    random_state=0
)
hist_model

HistGradientBoostingRegressor(early_stopping=True, max_iter=1000,
                              random_state=0)

We will use a grid-search to find some optimal parameter for this model. In
this grid-search, you should search for the following parameters:

* `max_depth: [3, 8]`;
* `max_leaf_nodes: [15, 31]`;
* `learning_rate: [0.1, 1]`.

Feel free to explore the space with additional values. Create the grid-search
providing the previous gradient boosting instance as the model.

In [3]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "max_depth": [3, 8],
    "max_leaf_nodes": [15, 31],
    "learning_rate": [0.1, 1]
}

search_cv = GridSearchCV(
    estimator=hist_model,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error"
)

Finally, we will run our experiment through cross-validation. In this regard,
define a 5-fold cross-validation. Besides, be sure to shuffle the data.
Subsequently, use the function `sklearn.model_selection.cross_validate` to run
the cross-validation. You should also set `return_estimator=True`, so that we
can investigate the inner model trained via cross-validation.

In [4]:
from sklearn.model_selection import KFold, cross_validate

# Define 5-fold cross-validation with shuffling
cv = KFold(n_splits=5, shuffle=True, random_state=0)

# Run cross-validation
cv_results = cross_validate(
    search_cv,
    data,
    target,
    cv=cv,
    scoring="neg_mean_absolute_error",
    return_estimator=True,
    n_jobs=-1
)

Now that we got the cross-validation results, print out the mean and standard deviation score.

In [5]:
print("Mean test score:", cv_results["test_score"].mean())
print("Standard deviation test score:", cv_results["test_score"].std())

Mean test score: -30.470151143235313
Standard deviation test score: 0.29635989260138446


Then inspect the estimator entry of the results and check the best parameters values. Besides, check the number of trees used by the model.


In [6]:
import pandas as pd

rows = []
for est in cv_results["estimator"]:
    row = est.best_params_.copy()
    row["n_trees"] = est.best_estimator_.n_iter_
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df.index.name = "fold"
results_df

,learning_rate,max_depth,max_leaf_nodes,n_trees
fold,,,,
0,0.1,8,31,214
1,0.1,8,15,447
2,0.1,8,15,423
3,0.1,8,15,290
4,0.1,8,31,145


Inspect the results of the inner CV for each estimator of the outer CV. Aggregate the mean test score for each parameter combination and make a box plot of these scores.


In [7]:
index_columns = [f"param_{name}" for name in param_grid.keys()]

inner_cv_results = []

for estimator in cv_results["estimator"]:
    df = pd.DataFrame(estimator.cv_results_)
    df = df.set_index(index_columns)[["mean_test_score"]]
    inner_cv_results.append(-df["mean_test_score"])

inner_cv_results = pd.concat(inner_cv_results, axis=1).T
inner_cv_results

param_max_depth               3                     8                     3  \
param_max_leaf_nodes         15         31         15         31         15   
param_learning_rate         0.1        0.1        0.1        0.1        1.0   
mean_test_score       45.363795  45.363795  44.418063  44.024151  51.784022   
mean_test_score       44.983669  44.983669  42.684050  44.082431  48.438538   
mean_test_score       44.371186  44.371186  43.708911  44.348723  51.082222   
mean_test_score       46.687255  46.687255  44.976488  44.980240  51.690035   
mean_test_score       46.133727  46.133727  44.111906  43.745421  50.199512   

param_max_depth                          8             
param_max_leaf_nodes         31         15         31  
param_learning_rate         1.0        1.0        1.0  
mean_test_score       51.784022  52.357514  55.777035  
mean_test_score       48.438538  52.913829  54.148831  
mean_test_score       51.082222  52.824518  54.231546  
mean_test_score       51.690035  52.151477  54.995510  
mean_test_score       50.199512  53.214412  53.559586